In [0]:
catalogo = "workspace"
ruta_aterrizaje = f"/Volumes/{catalogo}/riesgo_bronze/aterrizaje"
ruta_control = f"/Volumes/{catalogo}/riesgo_bronze/control"


In [0]:
from pyspark.sql import functions as F

ENTIDADES = {"clientes": "json", "solicitudes": "json", "prestamos": "csv", "pagos": "csv"}

def ingerir(entidad: str, formato: str):
    """Carga a Bronze los archivos nuevos de una entidad."""
    # aquí va tu bloque de ingesta: usa `entidad` y `formato`
    (spark.readStream
    .format("cloudFiles")                                   # activa Auto Loader
    .option("cloudFiles.format", f"{formato}")                     # formato de los archivos
    .option("cloudFiles.schemaLocation", f"{ruta_control}/esquemas/{entidad}")
    .option("cloudFiles.inferColumnTypes", "false")         # todo como texto
    .option("header", "true")                               # los CSV traen encabezado
    .load(f"{ruta_aterrizaje}/{entidad}")
    .withColumn("_archivo_origen", F.col("_metadata.file_path"))
    .withColumn("_ingerido_en", F.current_timestamp())
  .writeStream
    .option("checkpointLocation", f"{ruta_control}/checkpoints/{entidad}")
    .trigger(availableNow=True)                             # procesa lo pendiente y termina
    .toTable(f"{catalogo}.riesgo_bronze.{entidad}")
    .awaitTermination())

for entidad, formato in ENTIDADES.items():
    ingerir(entidad, formato)
    print(f"{entidad}: lista")



clientes: lista
solicitudes: lista
prestamos: lista
pagos: lista


In [0]:
ENTIDADES = {"clientes", "solicitudes", "prestamos", "pagos"}
for entidad in ENTIDADES:
    print(f"{entidad}: {spark.sql(f"SELECT count(*) as filas FROM workspace.riesgo_bronze.{entidad}").collect()[0]['filas']}")




clientes: 2806
prestamos: 1554
pagos: 1491
solicitudes: 3641
